# Автоматизация расчетов ключевых метрик Яндекс Такси


- Автор: Мельников Даниил
- Дата: 04.11.2026 г.

## Описание проекта

Задача — построить витрину данных, которая будет агрегировать информацию о поездках по каждому способу оплаты. Для этого нужно написать PySpark-скрипт, который рассчитает ключевые показатели: количество поездок, среднюю стоимость поездки, средние чаевые и суммарную выручку по каждому типу оплаты.

Чтобы процесс был полностью автоматическим и не зависел от ручных запусков, необходимо создать DAG в Airflow. Этот DAG должен ежедневно:

* проверять наличие новых файлов с данными;

* запускать Spark-задачу;

* формировать обновлённую итоговую таблицу.

Эта таблица станет основой для финансовых отчётов и аналитических дашбордов.

## Описание данных

Таблица `taxi_data` содержит данные об активности пользователей и состоит из следующих полей:

* `taxi_id` — идентификатор водителя;

* `trip_start_timestamp` — время начала поездки;

* `trip_end_timestamp` — время окончания поездки;

* `trip_seconds` — длительность поездки в секундах;

* `trip_miles` — дистанция поездки;

* `fare` — стоимость поездки;

* `tips` — размер чаевых;

* `trip_total` — общая стоимость поездки: стоимость поездки + чаевые + комиссия;

* `payment_type` — способ оплаты.

## Пайплайн работы над проектом

Для автоматизации сборки витрины данных по поездкам Яндекс Такси:

1. Сначала напишем Spark-скрипт, который будет обрабатывать данные о поездках и агрегировать показатели по способам оплаты `payment_type`. Рассчитаем несколько показателей:
* количество поездок, которое показывает общий спрос и загрузку сервиса;
* среднюю стоимость `fare`, которое отражает уровень среднего чека поездки;
* средние чаевые `tips` — индикатор удовлетворённости клиентов и мотивации водителей;
* суммарную выручку `trip_total` — ключевой показатель дохода компании.

Все результаты соберем в одну итоговую таблицу `taxi_payment_summary`. После этого запишем таблицу в ClickHouse с помощью JDBC-драйвера.

2. Далее настроим DAG в Airflow. Он запускается ежедневно, перед запуском он проверяет наличие файла с данными за нужную дату в S3-хранилище и только после появления файла запускает Spark-задачу.

## Шаг 1. Настраиваем Spark-агрегацию

In [ ]:
#filename=taxi_spark_job.py

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Создаем spark-сессию
spark = (
SparkSession.builder.appName("myAggregateTest")
.config("fs.s3a.endpoint", "storage.yandexcloud.net")
.config("spark.dynamicAllocation.enabled", "false")
.config("spark.executor.instances", "1")
.config("spark.executor.cores", "1")
.config("spark.executor.memory", "1g")
.config("spark.driver.memory", "1g")
.getOrCreate()
)

# Указываем порт и параметры кластера ClickHouse
jdbcPort = ...
jdbcHostname = "..."
username = "..."
jdbcDatabase = "..."
jdbcUrl = f"jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true"

# Считываем данные из файла типа parquet
parquet_path = "/taxi_data.parquet"

df = spark.read.parquet(parquet_path)

# Агрегируем по способу оплаты и считаем метрики
result_df = df.groupby("payment_type").agg(
    # Количество поездок:
    F.count(*).alias("trip_count"),
    # Средняя стоимость поездки:
    F.avg("fare").alias("avg_fare"),
    # Средний размер чаевых:
    F.avg("tips").alias("avg_tips"),
    # Суммарная выручка:
    F.sum("trip_total").alias("sum_trip_total")
)

# Записываем данные в таблицу, используем .mode("overwrite") для того, чтобы каждый раз объект перезаписывался, а не создавался заново
result_df.write.format("jdbc") \
    .option("url", jdbcUrl) \
    .option("user", username) \
    .option("password", "...") \
    .option("db_table", "taxi_payment_summary") \
    .option("createTableOptions", "ENGINE=MergeTree() ORDER BY payment_type") \
    .mode("overwrite") \
    .save()

## Шаг 2. Настраиваем DAG

In [ ]:
#filename=taxi_spark_job.py

from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

# Создаем DAG
DAG_ID = "taxi_payment_aggregate"

with DAG(
    DAG_ID,
    # Логическая дата старта:
    start_date=datetime(2025, 1, 1),
    # Запускается ежедневно:
    schedule_interval="@daily",
    # Запускаем DAG с текущего момента:
    catchup=False
) as dag:
    # Ждем появление файла в S3, для этого используем сенсор
    wait_for_s3 = S3KeySensor(
        task_id="wait_for_file",
        bucket_name="da-plus-dags",
        bucket_key="project_04/taxi_data.parquet",
        aws_conn_id="aws_default",
        # Проверяем наличие файла каждые 5 минут:
        poke_interval=300,
        # Максимальное время ожидания - 1 час:
        timeout=3600,
        mode="poke",
        # Ищем точное совпадение с бакетом
        wildcard_match=False
    )

    # Запускаем spark-джобу
    run_pyspark = DataprocCreatePysparkJobOperator(
        task_id="create_pyspark_job",
        name="aggregate_and_load",
        cluster_id="...",
        main_python_file_uri="/taxi_spark_job.py"
    )
    
    # Устанавливаем зависимости
    wait_for_s3 >> run_pyspark


## Шаг 3. Запускаем DAG с помощью Airflow UI

После этого данные будут записываться в таблицу согласно заданным параметрам и регулярно обновляться.